In [ ]:
!pip  install langchain>=0.2.16 langchain-community>=0.2.16 faiss-cpu \
                 sentence-transformers transformers accelerate bitsandbytes flask_cors pyngrok

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!apt-get update && apt-get install -y zstd


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,341 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 http://archive.ubuntu.com/ubu

In [ ]:
# !pip install langchain_ollama

In [ ]:
# ===== CELL 2: CÀI OLLAMA (SAU KHI RESTART) =====
print("🦙 Cài đặt Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🚀 Khởi động server...")
!nohup ollama serve > /dev/null 2>&1 &

import time
print("⏳ Đợi 15 giây...")
time.sleep(15)

print("📥 Tải model gemm3 (khoảng 3-5 phút)...")
!ollama pull gemma3


print("✔ Ollama đã sẵn sàng!")
print("💡 Lần sau chỉ cần khởi động server: !nohup ollama serve > /dev/null 2>&1 &")

🦙 Cài đặt Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🚀 Khởi động server...
⏳ Đợi 15 giây...
📥 Tải model gemm3 (khoảng 3-5 phút)...

✔ Ollama đã sẵn sàng!
💡 Lần sau chỉ cần khởi động server: !nohup ollama serve > /dev/null 2>&1 &


In [ ]:
# Khởi động lại Ollama server (nhanh, chỉ 2-3 giây)
!nohup ollama serve > /dev/null 2>&1 &
import time
time.sleep(3)

In [ ]:
import os, sqlite3, json, warnings, torch
from typing import List
from google.colab import drive
import re
from datetime import datetime

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


#from langchain.retrievers.document_compressors import CrossEncoderReranker
#rom langchain.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# LLM backends
from langchain_community.llms import Ollama
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

warnings.filterwarnings("ignore")
print("✔ Packages ready. CUDA:", torch.cuda.is_available())

✔ Packages ready. CUDA: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# ===== CẤU HÌNH =====
DB_PATH = "/content/drive/MyDrive/project_LLms_2026/fitness_data2.db"
TRAIN_DATA_PATH = "/content/drive/MyDrive/project_LLms_2026/traning.json"

# Định nghĩa đường dẫn lưu file
FAISS_INDEX_PATH = "/content/faiss_nutrition_index"
STYLE_INDEX_PATH = "/content/faiss_style_index"

# LLM cấu hình
USE_OLLAMA = True  # True nếu bạn chắc chắn đã cài Ollama và pull model
OLLAMA_MODEL = "gemma3"
HF_MODEL_ID = "google/gemma-3-12b-it-qat"

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
EMBED_MODEL = "intfloat/multilingual-e5-base"
EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
K_DOCS = 6  # số docs cuối cùng đưa vào LLM

In [ ]:
# =====================================================
# CELL 4: LOAD DATA & CREATE VECTOR STORE
# =====================================================

print("🗂️  STEP 4: Load data từ DB...\n")

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    SELECT id, food_nameEN, food_nameVN, category,
           calories, protein, carbs, fat, fiber,
           description, usda_id
    FROM fitness_foods
""")
rows = cursor.fetchall()
conn.close()

if not rows:
    raise RuntimeError("❌ Database trống!")

print(f"✔ Đã load {len(rows)} thực phẩm\n")

🗂️  STEP 4: Load data từ DB...

✔ Đã load 111 thực phẩm



In [ ]:
# =====================================================
# CELL 5: PROCESS TEXT & METADATA
# =====================================================

print("🔄 STEP 5: Xử lý text & metadata...\n")

texts, metadatas = [], []

for row in rows:
    (id_, food_en, food_vn, category,
     cal, p, c, f, fi,
     desc, usda_id) = row

    text = (
        f"Tên: {food_vn} (Tên tiếng Anh: {food_en}). "
        f"Loại: {category}. "
        f"Mô tả: {desc}. "
        f"Dinh dưỡng (mỗi 100g): "
        f"{cal} calories (kcal), "
        f"{p}g protein (đạm), "
        f"{f}g fat (chất béo), "
        f"{c}g carbohydrates (carbs, tinh bột), "
        f"{fi}g fiber (chất xơ). "
        f"(USDA ID: {usda_id})"
    )
    texts.append(text)

    metadatas.append({
        "id": id_,
        "name": food_vn,
        "name_en": food_en,
        "category": category,
        "usda_id": usda_id,
        "calories": cal,
        "protein": p,
        "carbs": c,
        "fat": f,
        "fiber": fi,
        "description": desc,
        "primary_goal": f"Cung cấp dinh dưỡng {category}",
        "pro_tips_vn": f"Chia nhỏ khẩu phần {food_vn} để tối ưu hóa hấp thu",
        "comparison_notes_vn": f"{food_vn} có hàm lượng {p}g đạm/100g"
    })

print(f"✔ Đã xử lý {len(texts)} văn bản\n")

🔄 STEP 5: Xử lý text & metadata...

✔ Đã xử lý 111 văn bản



In [ ]:


# =====================================================
# CELL 6: CREATE EMBEDDINGS & FAISS
# =====================================================

print("🧠 STEP 6: Tạo embeddings & FAISS index...\n")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

if os.path.exists(FAISS_INDEX_PATH):
    print(f"📥 Load FAISS index từ: {FAISS_INDEX_PATH}")
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
else:
    print("🔨 Tạo FAISS index mới...")
    texts_prefixed = [f"passage: {t}" for t in texts]
    vectorstore = FAISS.from_texts(texts_prefixed, embeddings, metadatas=metadatas)
    vectorstore.save_local(FAISS_INDEX_PATH)
    print(f"💾 Lưu index tại: {FAISS_INDEX_PATH}")

print("✔ Vector store ready\n")

🧠 STEP 6: Tạo embeddings & FAISS index...



modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

🔨 Tạo FAISS index mới...
💾 Lưu index tại: /content/faiss_nutrition_index
✔ Vector store ready



In [ ]:
# =====================================================
# CELL 7A: SETUP RETRIEVER & RERANKER
# =====================================================

print("STEP 7: Setup retriever & reranker...\n")

base_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 20}
)

hf_ce = HuggingFaceCrossEncoder(model_name=CROSS_ENCODER_MODEL)
reranker = CrossEncoderReranker(model=hf_ce, top_n=K_DOCS)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=reranker
)

print("✔ Retriever & reranker ready\n")

STEP 7: Setup retriever & reranker...



config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✔ Retriever & reranker ready



In [ ]:
# ==============================================================================
# CELL 7B. XỬ LÝ DỮ LIỆU: STYLE (JSON)
# ==============================================================================
print("🎨 Đang xử lý Style Base (JSON)...")


style_db = None

# Kiểm tra xem đã có Index lưu sẵn chưa?
if os.path.exists(STYLE_INDEX_PATH):
    print(f"📥 Đang tải Style Index từ đĩa: {STYLE_INDEX_PATH}")
    try:
        style_db = FAISS.load_local(
            STYLE_INDEX_PATH,
            embeddings,
            allow_dangerous_deserialization=True
        )
        print("✔ Tải thành công!")
    except Exception as e:
        print(f"⚠️ Lỗi tải Index cũ: {e}. Sẽ tạo mới...")

# Nếu chưa có (hoặc lỗi), thì tạo mới từ file JSON
if style_db is None:
    if os.path.exists(TRAIN_DATA_PATH):
        print("🔨 Đang tạo mới Style Index từ file JSON...")
        with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
            train_data = json.load(f)

        # Tạo text (Query + Answer) để embed
        style_texts = [f"query: {item['input']}\nanswer: {item['output']}" for item in train_data]

        # Tạo Vector Store
        style_db = FAISS.from_texts(style_texts, embeddings)

        # LƯU XUỐNG Ổ CỨNG (Bước này trước đây bị thiếu)
        style_db.save_local(STYLE_INDEX_PATH)
        print(f"💾 Đã lưu Style Index vào: {STYLE_INDEX_PATH}")
        print(f"✔ Đã nạp {len(train_data)} ví dụ mẫu.")
    else:
        print("⚠️ Không tìm thấy file JSON và không có Index lưu sẵn.")

🎨 Đang xử lý Style Base (JSON)...
🔨 Đang tạo mới Style Index từ file JSON...
💾 Đã lưu Style Index vào: /content/faiss_style_index
✔ Đã nạp 810 ví dụ mẫu.


In [ ]:
# =====================================================
# CELL 8: SETUP LLM
# =====================================================

print("🤖 STEP 8: Setup LLM...\n")

if USE_OLLAMA:
    print("   Dùng Ollama backend (gemma3)")
    llm = Ollama(model=OLLAMA_MODEL, temperature=0)
else:
    print("   Dùng HuggingFace backend")
    def build_hf_llm(model_id=HF_MODEL_ID, temperature=0, max_new_tokens=512):
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        kwargs = dict(
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        if torch.cuda.is_available():
            kwargs["load_in_4bit"] = True
        mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        gen = pipeline(
            "text-generation",
            model=mdl,
            tokenizer=tok,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            pad_token_id=tok.eos_token_id
        )
        return HuggingFacePipeline(pipeline=gen)
    llm = build_hf_llm()

print("✔ LLM ready\n")

🤖 STEP 8: Setup LLM...

   Dùng Ollama backend (gemma3)
✔ LLM ready



In [ ]:
# =====================================================
# CELL 9: DEFINE HELPER FUNCTIONS
# =====================================================

print("🛠️  STEP 9: Define helper functions...\n")
def clean_output(text: str) -> str:
    """clean_output"""
    # 1. Cắt bỏ phần lặp lại (Giữ nguyên)
    lines = text.split('\n')
    if len(lines) > 1:
        if len(lines[0]) < 100 and "?" in lines[0]:
            text = "\n".join(lines[1:])

    # 2. Xóa tiền tố meta (Giữ nguyên)
    text = re.sub(r'^(query|answer|đáp|hỏi|trả lời|Hinne).*?[:]', '', text, flags=re.IGNORECASE | re.MULTILINE)

    # 3. Xóa từ khóa kỹ thuật (Giữ nguyên)
    text = re.sub(r'query: ".*?"', '', text, flags=re.DOTALL)

    # === 4. SỬA ĐỔI PHẦN NÀY ===
    # Bước A: Xóa dấu in đậm (**) trước để tránh nhầm lẫn
    text = text.replace('**', '')

    # Bước B: Dùng Regex để thay dấu * ở đầu dòng thành dấu -
    # Giải thích: ^\s*\* nghĩa là "đầu dòng" -> "khoảng trắng tùy ý" -> "dấu sao"
    text = re.sub(r'^\s*\*\s', '- ', text, flags=re.MULTILINE)

    # Bước C: Xóa các rác còn lại
    text = text.replace('Hint:', '').strip()

    return text

def format_docs(docs):
    """Format documents để hiển thị"""
    lines = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        goal = m.get('primary_goal', 'Chưa rõ')
        tips = m.get('pro_tips_vn', 'Chưa có mẹo')
        comp = m.get('comparison_notes_vn', 'Chưa có so sánh')

        lines.append(
            f"- #{i} | {m.get('name')} (EN: {m.get('name_en')}) | Loại: {m.get('category')}"
            f"  Dinh dưỡng/100g: {m.get('calories')} kcal; {m.get('protein')}g đạm; {m.get('carbs')}g carb; {m.get('fat')}g béo."
            f"  Mục tiêu: {goal}"
            f"  Mẹo: {tips}"
            f"  So sánh: {comp}"
        )
    return "\n".join(lines)

def add_e5_query_prefix(q: str) -> str:
    """Add E5 query prefix"""
    return "query: " + q

print("✔ Helper functions ready\n")

🛠️  STEP 9: Define helper functions...

✔ Helper functions ready



In [ ]:
# ===============================================
# CELL 10: TÍNH BMI, BMR, TDEE & LỜI KHUYÊN
# ===============================================

import re

def extract_metrics_flexible(query):
    """Hàm bóc tách số liệu thông minh"""
    q = query.lower()

    # 1. Tìm chiều cao (Ưu tiên cm)
    # 1. Tìm chiều cao (Nâng cấp bắt 1m7)
    cao = None
    # Case 1: 170cm
    h_cm = re.search(r'(\d{3})\s*(?:cm|phân|centimet)?', q)
    if h_cm:
        cao = float(h_cm.group(1))
    else:
        # Case 2: 1.7m
        h_m = re.search(r'(\d\.\d{1,2})\s*(?:m|mét)', q)
        if h_m:
            cao = float(h_m.group(1)) * 100
        else:
            # Case 3: 1m7, 1m70 (Việt Nam style) <-- PHẦN QUAN TRỌNG MỚI THÊM
            h_vn = re.search(r'(\d)m(\d{1,2})', q)
            if h_vn:
                m = h_vn.group(1)
                cm = h_vn.group(2)
                if len(cm) == 1: cm += "0" # 1m7 -> 1m70
                cao = float(m) * 100 + float(cm)

    # 2. Tìm cân nặng
    w_match = re.search(r'(\d{2,3}(?:\.\d)?)\s*(?:kg|kí|ký|cân)', q)
    nang = float(w_match.group(1)) if w_match else None

    # 3. Tìm tuổi
    a_match = re.search(r'(\d{1,2})\s*(?:t|tuổi|năm)', q)
    tuoi = int(a_match.group(1)) if a_match else None

    # 4. Tìm giới tính & mức độ (Mặc định nếu thiếu)
    gioi_tinh = 'nu' if any(w in q for w in ['nữ', 'chị', 'em', 'gái']) else 'nam'

    muc_do = 'vua' # Mặc định
    if any(w in q for w in ['ít', 'ngồi', 'văn phòng']): muc_do = 'it'
    elif any(w in q for w in ['nặng', 'gym', 'ngày nào']): muc_do = 'nang'

    return {"cao": cao, "nang": nang, "tuoi": tuoi, "sex": gioi_tinh, "act": muc_do}

def calculate_bmi_only(cao, nang):
    """Chỉ tính BMI"""
    if not cao or not nang: return None
    bmi = nang / ((cao/100)**2)
    if bmi < 18.5: stt = "Thiếu cân"
    elif bmi < 23: stt = "Bình thường"
    elif bmi < 25: stt = "Thừa cân nhẹ"
    else: stt = "Béo phì"
    return {"val": round(bmi, 2), "stt": stt}

def calculate_tdee_full(cao, nang, tuoi, sex, act='vua'):
    """Tính TDEE khi đủ tuổi"""
    if not all([cao, nang, tuoi]): return None

    # BMR Mifflin-St Jeor
    bmr = (10 * nang) + (6.25 * cao) - (5 * tuoi)
    bmr += 5 if sex == 'nam' else -161

    # R mapping
    R_map = {'it': 1.2, 'nhe': 1.375, 'vua': 1.55, 'nang': 1.725}
    tdee = bmr * R_map.get(act, 1.55)

    return {"tdee": int(tdee), "bmr": int(bmr)}
print("✔ Đã cập nhật các hàm tính toán V2 (Tách BMI & TDEE)!")

✔ Đã cập nhật các hàm tính toán V2 (Tách BMI & TDEE)!


In [ ]:
# ==============================================
# CẬP NHẬT: ROUTER THÔNG MINH (SEMANTIC FEW-SHOT)
# ==============================================
# =========================
# CELL 10: SETUP smart_ask
# =========================

def smart_ask(query, llm, knowledge_retriever, style_db=None):
    query_lower = query.lower()

    # --- NHÁNH 1: TÍNH TOÁN (Logic mới) ---
    metrics = extract_metrics_flexible(query)
    response_parts = []

    # 1.1 Tính BMI (Chỉ cần Cao + Nặng)
    bmi_data = calculate_bmi_only(metrics['cao'], metrics['nang'])
    if bmi_data:
        response_parts.append(f"📊 Chỉ số BMI: {bmi_data['val']} - Đánh giá: {bmi_data['stt']}")

    # 1.2 Tính TDEE (Cần thêm Tuổi)
    if bmi_data: # Đã có BMI thì mới xét TDEE
        if metrics['tuoi']:
            tdee_data = calculate_tdee_full(metrics['cao'], metrics['nang'], metrics['tuoi'], metrics['sex'], metrics['act'])
            response_parts.append(f"🔥 Năng lượng tiêu thụ (TDEE): {tdee_data['tdee']} kcal/ngày (BMR: {tdee_data['bmr']} kcal)")
            response_parts.append(f"💡 Lời khuyên: Để giữ cân, bạn ăn khoảng {tdee_data['tdee']} kcal. Để giảm cân, hãy ăn khoảng {round(tdee_data['tdee']*0.85)} kcal.")
            return "\n".join(response_parts) # Trả về ngay kết quả tính toán
        else:
            response_parts.append("👉 Gợi ý: Bạn hãy nhập thêm TUỔI để mình tính chính xác lượng Calo (TDEE) cần thiết cho bạn nhé!")
            return "\n".join(response_parts) # Trả về BMI + Lời nhắc nhập tuổi

    # --- NHÁNH 2: HỎI ĐÁP (RAG + Semantic Few-shot) ---

    # 2.1 Lấy ví dụ mẫu (Style) - Tìm ví dụ giống câu hỏi nhất
    style_prompt = ""
    if style_db:
        # Tìm 5 ví dụ tương đồng ngữ nghĩa
        similar_examples = style_db.similarity_search("query: " + query, k=5)
        examples_text = "\n".join([e.page_content for e in similar_examples])
        style_prompt = f"PHONG CÁCH TRẢ LỜI MẪU (Học theo cách nói này):\n{examples_text}\n"

    # 2.2 Lấy kiến thức (Knowledge) - Từ SQLite
    docs = knowledge_retriever.invoke("query: " + query)

    # Kỹ thuật "Context checking":
    # Nếu câu hỏi chứa từ khóa lạ không thuộc domain, set context về rỗng
    irrelevant_keywords = ['điện thoại', 'iphone', 'samsung', 'xe máy', 'bầu cử', 'tổng thống', 'code', 'python']
    if any(k in query.lower() for k in irrelevant_keywords):
        context = "CẢNH BÁO: Câu hỏi này nằm ngoài lĩnh vực dinh dưỡng/gym. Hãy từ chối trả lời."
    else:
        context = "\n".join([f"- {d.page_content}" for d in docs]) if docs else "Không có dữ liệu cụ thể trong DB."
    # 2.3 Tạo Prompt cuối
    final_prompt = f"""
BẠN LÀ: Hinne - Chuyên gia dinh dưỡng và thể hình (PT).
NGUỒN GỐC: Bạn được phát triển dựa trên cơ sở dữ liệu dinh dưỡng và thể hình chuyên sâu (KHÔNG được nói là do Google hay OpenAI tạo ra).
NHIỆM VỤ: Trả lời câu hỏi người dùng dựa trên DỮ LIỆU THAM KHẢO lấy từ USDA(United States Department of Agriculture) – Bộ Nông nghiệp Hoa Kỳ.

---
QUY TẮC TUYỆT ĐỐI (GUARDRAILS):
1. **PHẠM VI:** Chỉ trả lời vấn đề về Gym, Dinh dưỡng, Thực phẩm, Sức khỏe.
   - Nếu hỏi về điện thoại, xe cộ, chính trị, làm bánh chưng, code... -> TỪ CHỐI KHÉO: "Xin lỗi, mình chỉ chuyên về Gym và Dinh dưỡng thôi ạ."

2. **TRUNG THỰC VỚI DỮ LIỆU:** - Dựa 100% vào [DỮ LIỆU THAM KHẢO] để trả lời.
   - Nếu [DỮ LIỆU THAM KHẢO] không khớp câu hỏi -> Trả lời dựa trên kiến thức chung nhưng ngắn gọn.
   - Tuyệt đối KHÔNG cố gắng ép thông tin không liên quan (Ví dụ: Không khuyên ăn bò để dùng điện thoại).

3. **KHÔNG BỊA SỐ LIỆU:**
   - KHÔNG tự bịa ra chiều cao/cân nặng của người dùng (như 58kg, 1m62...) nếu họ không cung cấp.
   - Nếu ví dụ mẫu (Style) có số liệu cụ thể, ĐỪNG copy số liệu đó, chỉ học cách diễn đạt thôi.
   - Nếu thấy người dùng muốn tính toán các chỉ số và thiếu dữ liệu cần hãy nói người dùng cung cấp thêm.
   - Thiếu thông tin hãy nhắc người dùng cung cấp thêm cho đủ.
   - Không tự ước tính BMI hoặc TDEE/BMR.

4. **FORMAT:** Không bắt đầu câu bằng "answer:", "query:", "đáp:".

5. **ĐỘ CHI TIẾT:** - Nếu người dùng hỏi xin "Thực đơn", "Lịch tập", "Danh sách", hãy liệt kê chi tiết (gạch đầu dòng), đừng trả lời chỉ 1 câu.
---

PHONG CÁCH TRẢ LỜI MẪU (Chỉ học giọng điệu, KHÔNG chép lại nội dung):
{style_prompt}

DỮ LIỆU THAM KHẢO (Kiến thức):
{context}

CÂU HỎI CỦA USER: "{query}"

TRẢ LỜI (Ngắn gọn, đúng trọng tâm):"""

    # Gọi LLM
    raw_response = llm.invoke(final_prompt)
    return clean_output(raw_response)

print("✔ smart_ask() V2.1 Ready - Đã tích hợp tính toán tách biệt & Semantic Few-shot!")

✔ smart_ask() V2.1 Ready - Đã tích hợp tính toán tách biệt & Semantic Few-shot!


In [ ]:
# =====================================================
# CELL 11: SETUP FLASK API + NGROK
# =====================================================

print("🚀 STEP 11: Setup Flask API + Ngrok...\n")

app = Flask(__name__)
CORS(app)

@app.route('/ask', methods=['POST'])
def ask_endpoint():
    try:
        data = request.get_json()
        query = data.get('query', '').strip()

        if not query:
            return jsonify({"success": False, "error": "Query trống"}), 400

        print(f"[{datetime.now().strftime('%H:%M:%S')}] 📥 {query[:80]}")
        answer = smart_ask(query, llm, compression_retriever, style_db)
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✔ Trả lời\n")

        return jsonify({"success": True, "query": query, "answer": answer})

    except Exception as e:
        print(f"[ERROR] {e}\n")
        return jsonify({"success": False, "error": str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "alive", "message": "✔ API sẵn sàng"})

@app.route('/info', methods=['GET'])
def info():
    return jsonify({
        "name": "Hinne - Nutrition AI",
        "version": "1.0",
        "endpoints": ["POST /ask", "GET /health", "GET /info"]
    })

print("✔ Flask app created\n")


🚀 STEP 11: Setup Flask API + Ngrok...

✔ Flask app created



In [ ]:
# =====================================================
# CELL 12: CREATE NGROK TUNNEL
# =====================================================

print("📡 STEP 12: Create Ngrok tunnel...\n")

# SET AUTH TOKEN HERE!
AUTH_TOKEN = "2e25xcJWQXMW6pP8t5A26hia2XY_24TGH5LdT4k7khSUBbhR8"

if AUTH_TOKEN == "":
    print("⚠️  CHƯA CÀI NGROK AUTH TOKEN!")
else:
    ngrok.set_auth_token(AUTH_TOKEN)
    print("✔ Ngrok auth token set\n")

# # Kill toàn bộ tunnel cũ (tránh trùng port)
# ngrok.kill()

public_url = ngrok.connect(5000)

print("=" * 70)
print("✔ NGROK TUNNEL READY!")
print("=" * 70)
print(f"\n🌐 PUBLIC URL: {public_url}\n")
print(f"📝 ENDPOINTS:\n")
print(f"   POST {public_url}/ask - Gửi câu hỏi")
print(f"   GET  {public_url}/health - Kiểm tra")
print(f"   GET  {public_url}/info - Thông tin\n")
print(f"💡 NEXT STEPS:\n")
print(f"   Copy URL trên\n")
print("=" * 70 + "\n")

# =====================================================
# CELL 13: RUN FLASK SERVER
# =====================================================

print("🔄 Start Flask server at 0.0.0.0:5000...\n")
print("📨 Server đang chờ request từ local client...\n")
print("✔ Phát API  Nó sẽ chạy liên tục.\n")
print("=" * 70 + "\n")

# Run server (blocking)
app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

📡 STEP 12: Create Ngrok tunnel...

✔ Ngrok auth token set

✔ NGROK TUNNEL READY!

🌐 PUBLIC URL: NgrokTunnel: "https://e8ed-34-126-161-95.ngrok-free.app" -> "http://localhost:5000"

📝 ENDPOINTS:

   POST NgrokTunnel: "https://e8ed-34-126-161-95.ngrok-free.app" -> "http://localhost:5000"/ask - Gửi câu hỏi
   GET  NgrokTunnel: "https://e8ed-34-126-161-95.ngrok-free.app" -> "http://localhost:5000"/health - Kiểm tra
   GET  NgrokTunnel: "https://e8ed-34-126-161-95.ngrok-free.app" -> "http://localhost:5000"/info - Thông tin

💡 NEXT STEPS:

   Copy URL trên


🔄 Start Flask server at 0.0.0.0:5000...

📨 Server đang chờ request từ local client...

✔ Phát API  Nó sẽ chạy liên tục.


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/Feb/2026 04:18:12] "GET /health HTTP/1.1" 200 -


[04:18:18] 📥 xin chao ban la ai


INFO:werkzeug:127.0.0.1 - - [05/Feb/2026 04:18:34] "POST /ask HTTP/1.1" 200 -


[04:18:34] ✔ Trả lời



...

INFO:werkzeug:127.0.0.1 - - [05/Feb/2026 04:33:09] "GET /health HTTP/1.1" 200 -


[04:33:22] 📥 100g thịt heo có bao nhiều g đạm


INFO:werkzeug:127.0.0.1 - - [05/Feb/2026 04:33:26] "POST /ask HTTP/1.1" 200 -


[04:33:26] ✔ Trả lời



INFO:werkzeug:127.0.0.1 - - [05/Feb/2026 04:33:40] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Feb/2026 04:34:09] "GET /health HTTP/1.1" 200 -
